# Notebook 2 - Measurements and State Estimation

In this notebook, we implement a function to create measurements in the network. You will then modify the measurement set to compare state estimation results.

## 2.1 Set up
Network and libraries import

In [ ]:
import copy
import pandapower as pp
import pandas as pd
import numpy as np
from pandapower.estimation import estimate

net = pp.from_json('data/svedala.json')
print(net)



## 2.2 Measurement function creation
We iteratively create measurements on different network components.

In [ ]:
def create_measurement_vectors(net, v_stddev=0.004, p_stddev=50, q_stddev=5,
                                 add_noise=False, noise_zones=None,zero_injection_buses=[14,22,27,28]):
    
    """ Create measurement vectors for voltage magnitudes, active and reactive power injections, 
    and active and reactive power flows.
    !! It reinitializes the net.measurement DataFrame 
    # No measurements are created at zero injection buses
    """

    np.random.seed(5) # Ensure replicability (i.e. ensure that the same set of measurements is generated for the same stds)
    if noise_zones is not None:
        if isinstance(noise_zones, int):
            noise_zones = [noise_zones]
        elif not isinstance(noise_zones, list):
            noise_zones = None
    
    net.measurement = net.measurement[0:0] # Clear existing measurements
    pp.runpp(net, enforce_q_lims=True) # Run the loadflow
    
    # BUSES 
    for busIndex in net.bus.index:     
        bus_zone = net.bus.loc[busIndex, 'zone']
        apply_noise = add_noise and (noise_zones is None or bus_zone in noise_zones)
            
        # Voltage measurement
        real_val = net.res_bus.vm_pu[busIndex]
        meas_val = np.random.normal(real_val, v_stddev) if apply_noise else real_val
        pp.create_measurement(net, "v", "bus", meas_val, v_stddev, element=busIndex)
            
        # Power injections
        real_val_p = net.res_bus.p_mw[busIndex]
        real_val_q = net.res_bus.q_mvar[busIndex]    
            
        if not (busIndex in zero_injection_buses):  # Do not create measurements for zero injection buses            
            if real_val_p != 0:
                meas_val = np.random.normal(real_val_p, p_stddev) if apply_noise else real_val_p
                pp.create_measurement(net, "p", "bus", meas_val, p_stddev, element=busIndex)                
            if real_val_q != 0:
                meas_val = np.random.normal(real_val_q, q_stddev) if apply_noise else real_val_q
                # pp.create_measurement(net, "q", "bus", meas_val, q_stddev, element=busIndex)
    
    # LINES           
    for lineIndex in net.line.index:    
        if net.line.loc[lineIndex, 'in_service']:        
            lineIndex = int(lineIndex)
            from_bus = net.line.loc[lineIndex, 'from_bus']
            to_bus = net.line.loc[lineIndex, 'to_bus']
            from_zone = net.bus.loc[from_bus, 'zone']
            to_zone = net.bus.loc[to_bus, 'zone']
            apply_noise = add_noise and (noise_zones is None or from_zone in noise_zones or to_zone in noise_zones)
                
            for side in ['from', 'to']:                             
                real_val = net.res_line.loc[lineIndex, f'p_{side}_mw']
                meas_val = np.random.normal(real_val, p_stddev) if apply_noise else real_val
                pp.create_measurement(net, "p", "line", meas_val, p_stddev, element=lineIndex, side=side)
                        
                real_val = net.res_line.loc[lineIndex, f'q_{side}_mvar']
                meas_val = np.random.normal(real_val, q_stddev) if apply_noise else real_val
                # pp.create_measurement(net, "q", "line", meas_val, q_stddev, element=lineIndex, side=side)
    
    # TRANSFORMERS
    for trafoIndex in net.trafo.index:
        if net.trafo.loc[trafoIndex, 'in_service']:
            trafoIndex = int(trafoIndex)
            hv_bus = net.trafo.loc[trafoIndex, 'hv_bus']
            lv_bus = net.trafo.loc[trafoIndex, 'lv_bus']
            hv_zone = net.bus.loc[hv_bus, 'zone']
            lv_zone = net.bus.loc[lv_bus, 'zone']
            apply_noise = add_noise and (noise_zones is None or hv_zone in noise_zones or lv_zone in noise_zones)
                
            for side in ['hv', 'lv']:
                real_val = net.res_trafo.loc[trafoIndex, f'p_{side}_mw']
                meas_val = np.random.normal(real_val, p_stddev) if apply_noise else real_val
                pp.create_measurement(net, "p", "trafo", meas_val, p_stddev, element=trafoIndex, side=side)
                    
                real_val = net.res_trafo.loc[trafoIndex, f'q_{side}_mvar']
                meas_val = np.random.normal(real_val, q_stddev) if apply_noise else real_val
                # pp.create_measurement(net, "q", "trafo", meas_val, q_stddev, element=trafoIndex, side=side)       

## 2.3 Running the state estimation

We can now create measurements.

In [ ]:
create_measurement_vectors(net)
net.measurement.head(5)

Rune the state estimation.

In [ ]:
success = estimate(net, init='flat', algorithm='wls', tolerance=1e-6, maximum_iterations=10)
print(f'Estimate call returned: {success}')

Show the results.

In [ ]:
print("Buses power flow results:")
net.res_bus[["vm_pu", "va_degree"]].head(5)

In [ ]:
print("Buses estimated results:")
net.res_bus_est[["vm_pu", "va_degree"]].head(5)

In [ ]:
print("Lines power flow results:")
net.res_line[["p_from_mw", "q_from_mvar"]].head(5)

In [ ]:
print("Lines estimated results:")
net.res_line_est[["p_from_mw", "q_from_mvar"]].head(5)

## 2.4 Checking residuals
In practice, you do not have access to the real results. Instead, you can compare the estimates with with the measured values by computing the residuals.



In [ ]:
def compute_all_residuals(net):
    """
    Compute residuals for all measurements.
    """
    meas = net.measurement.copy()
    results = []

    for idx, m in meas.iterrows():
        m_type = m["measurement_type"]
        el_type = m["element_type"]
        el = m["element"]

        value = m["value"]
        std = m["std_dev"]

        # Buses
        if el_type == "bus":
            if m_type == "v":
                est = net.res_bus_est.vm_pu.loc[el]
            elif m_type == "p":
                est = net.res_bus_est.p_mw.loc[el]
            elif m_type == "q":
                est = net.res_bus_est.q_mvar.loc[el]
            else:
                continue

        # Lines
        elif el_type == "line":
            if m_type == "p":
                est = net.res_line_est.loc[el, f"p_{m['side']}_mw"]
            elif m_type == "q":
                est = net.res_line_est.loc[el, f"q_{m['side']}_mvar"]
            else:
                continue

        # Trafos
        elif el_type == "trafo":
            if m_type == "p":
                est = net.res_trafo_est.loc[el, f"p_{m['side']}_mw"]
            elif m_type == "q":
                est = net.res_trafo_est.loc[el, f"q_{m['side']}_mvar"]
            else:
                continue
        else:
            continue

        residual = value - est       

        results.append({
            "measurement_type": m_type,
            "element_type": el_type,
            "element_id": el,
            "side": m.get("side", None),
            "value": value,
            "estimated": est,
            "residual": residual,
            "std": std
        })
    residual_table = pd.DataFrame(results)
    residual_table["label"] = (residual_table["measurement_type"].astype(str)+ "_"+ residual_table["element_type"].astype(str)+ "_"+ residual_table["element_id"].astype(str))
    return residual_table

residuals = compute_all_residuals(net)

We can also define functions to visualize these results.

In [ ]:
import matplotlib.pyplot as plt

def plot_residuals(df, title="Residuals", threshold=None, label_col=None):
    df = df.copy()
    if label_col and label_col in df.columns:
        labels = df[label_col]
    else:
        labels = df.index.astype(str)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(df)),
           df["residual"],
           color="steelblue",
           edgecolor="black",
           alpha=0.75)

    if threshold is not None:
        ax.axhline(threshold, color="red", linestyle="--")
        ax.axhline(-threshold, color="red", linestyle="--")

    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(labels, rotation=45, ha="right")

    ax.set_ylabel("Residual")
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_top_residuals(df, n=20, title="Largest residuals"):
    import matplotlib.pyplot as plt

    d = df.copy()
    d = d.sort_values(by="residual", key=lambda x: abs(x), ascending=False).head(n)

    labels = d.get("label", d.index.astype(str))

    plt.figure(figsize=(12, 5))
    plt.bar(range(len(d)), d["residual"], color="steelblue", edgecolor="black")

    plt.xticks(range(len(d)), labels, rotation=45, ha="right")
    plt.ylabel("Residual")
    plt.title(title)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_residuals_by_type(df):
    import matplotlib.pyplot as plt
    types = df["measurement_type"].unique()
    fig, ax = plt.subplots(figsize=(12, 5))
    for t in types:
        subset = df[df["measurement_type"] == t]
        ax.scatter([t]*len(subset), subset["residual"], alpha=0.4)

    ax.set_ylabel("Residual")
    ax.set_title("Residuals by measurement type")
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
plot_top_residuals(residuals, n=20)
plot_residuals_by_type(residuals)

## 2.5 Exercises
You can now try to modify the network or the measurement set and check of it impacts the results.

Suggested approaches are the following:

## A. Bad measurement
1. Modify or create measurements to implement bad measurements in the network, rerun the state estimation and check the residuals. Does it impact all residuals and the overall accuracy of the results? Can this error be spotted with the residuals?
2. Use the remove_bad_data function of pandapower and verify that it deletes your measurement. Does it delete other measurements?

## B. Topological change
Disconnect a line by setting "in_service" to false and rerun the state estimation. You can now follow the same procedure as in A.


In [ ]:
# A: Adding bad measurements and testing bad data detection
from pandapower.estimation import remove_bad_data
net2 = copy.deepcopy(net) # Creating a copy of the net to keep the original one for comparison

## Create measurements with a large error
true_val = net2.res_line.at[1, "p_from_mw"] # example
pp.create_measurement(net2, )
print(net2.measurement[["element_type", "element", "value", ]].tail(5)) # You can check the last few measurements to find yours

In [ ]:
# B: Adding a topological error
net_topo = copy.deepcopy(net)

# Disconnect a line

In [ ]:
from pandapower.estimation import remove_bad_data
net2 = copy.deepcopy(net)
success_rn_max = remove_bad_data(net2, init='flat', rn_max_threshold=20)
print(success_rn_max)
print(net2)
residuals2 = compute_all_residuals(net2)
plot_residuals(residuals2, title="Residuals after bad data removal")
pp.create_measurement(net2, "p", "line", 9, 0.5, element=1, side="from")
residuals2 = compute_all_residuals(net2)
plot_residuals(residuals2, title="Residuals after bad data removal")